# Правила

**Даты**: 22.12.2024 - 28.12.2024

**Дедлайн**: 28.12.2024 23:59

**Отправка после дедлайна**: НЕВОЗМОЖНА


За каждое задание вы получите определенное количество баллов (указано в скобках рядом с каждым заданием). Максимальная оценка за работу - 10 баллов.

* **Задания, где вы должны написать код**, помечены 🐴 (эмодзи лошади).

* **Вопросы, на которые вы должны дать ответ текстом**, помечены знаком ❓ (эмодзи вопроса).

Вы также увидите код "assert", этот код должен вам помочь: если вы все делаете правильно - код не выдаст ошибку AssertionError.

**Дисклеймер**: экзамен должен выполняться самостоятельно. "Очень похожие" решения считаются плагиатом, и все зайдествованные студенты (включая тех, у кого работа была скопирована) получают 0 баллов за экзамен.

# Подготовка

Обратите внимание, что вам необходимо импортировать дополнительные библиотеки и модули для решения.

In [ ]:
# Работа с данными
import pandas as pd
import numpy as np

# Визуализации
import matplotlib.pyplot as plt
import plotly.express as px

# Тесты
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller, kpss

# Декомпозиция
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.seasonal import STL as STL_decomp
from statsmodels.tsa.seasonal import MSTL as MSTL_decomp

# Модель Prophet
import prophet as fp

# Тюнинг
import itertools

In [ ]:
!pip install pmdarima

# авто ARIMA
from pmdarima import auto_arima

In [ ]:
!pip install statsforecast

# MSTL
from statsforecast import StatsForecast
from statsforecast.models import (
    MSTL
  )

In [ ]:
def adf_test(timeseries):
    print("Results of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=[
            "Test Statistic",
            "p-value",
            "#Lags Used",
            "Number of Observations Used",
        ],
    )
    for key, value in dftest[4].items():
        dfoutput["Critical Value (%s)" % key] = value
    print(dfoutput)

In [ ]:
def kpss_test(timeseries):
    print("Results of KPSS Test:")
    kpsstest = kpss(timeseries, regression="c", nlags="auto")
    kpss_output = pd.Series(
        kpsstest[0:3], index=["Test Statistic", "p-value", "Lags Used"]
    )
    for key, value in kpsstest[3].items():
        kpss_output["Critical Value (%s)" % key] = value
    print(kpss_output)

In [ ]:
def rmse(predictions, targets):
    return np.sqrt(((predictions - targets) ** 2).mean())


# Описание задачи

Вы будете работать над прогнозированием показателей *BookBnb*, платформы краткосрочной аренды, которую любят путешественники за ее услуги и качество объектов недвижимости. Вы будете прогнозировать метрику **просмотров** для объявлений (=объектов недвижимости) из *Калининграда*, представленных на платформе.

Метрика **просмотров** показывает, сколько раз пользователи просматривали объявление о недвижимости на платформе. Она служит показателем интереса или спроса к конкретному объекту недвижимости или месту назначения (например, городу). Этот показатель имеет решающее значение для понимания вовлеченности пользователей, привлекательности объявлений и общей активности на платформе. Таким образом, количество просмотров в Калининграде отражает уровень интереса к этому региону, что делает его ключевым показателем для оценки его популярности и потенциала для дальнейших инвестиций.

Основная цель - оценить популярность Калининграда как туристического направления на платформе и определить, стоит ли инвестировать в рекламные объекты в этом регионе. Ваш прогноз поможет BookBnb принимать стратегические решения о распределении ресурсов и маркетинговых кампаниях для стимулирования роста и улучшения взаимодействия с пользователями.

Ваша цель - составить надежный прогноз на следующий год и доказать, что компания может ему доверять. Этот прогноз послужит основой для принятия обоснованных решений на конкурентном рынке аренды. Удачи! 🚀

Обратите внимание на горизонт прогноза: ваша последняя задача - спрогнозировать один будущий год.

In [ ]:
FORECAST_HORIZON = 365

# Данные

Вы можете загрузить датасет *exam_views_kgd.csv* с временными рядами метрики *просмотров* по [ссылке](https://drive.google.com/file/d/1XRh0JKvFGtNjOJXG4_qZBDWX9R5FXldb/view?usp=sharing).

Датасет включает исторические данные **с начала 2019 года по конец 2022 года** (представьте, что сейчас 2022 год :D).

**Колонки:**

* *region* - регион, в котором присутствуют объекты недвижимости, в этом датасете есть только "Kaliningrad".;

* *dt* - даты, подневно, без пропусков;

* *metric* - название метрики, в этом датасете есть только "views";

* *value* - значение метрики.

In [ ]:
df = pd.read_csv('exam_views_kgd.csv', sep=';', parse_dates=['dt'])
df

,region,dt,metric,value
0,Kaliningrad,2019-01-01,views,26011.0
1,Kaliningrad,2019-01-02,views,23158.0
2,Kaliningrad,2019-01-03,views,23413.0
3,Kaliningrad,2019-01-04,views,22534.0
4,Kaliningrad,2019-01-05,views,21431.0
...,...,...,...,...
1456,Kaliningrad,2022-12-27,views,23796.0
1457,Kaliningrad,2022-12-28,views,22762.0
1458,Kaliningrad,2022-12-29,views,20338.0
1459,Kaliningrad,2022-12-30,views,10723.0


Let's plot the time series.

In [ ]:
fig = px.line(title="BookBnb views for listings from Kaliningrad, daily")
fig.add_scatter(x=df.dt, y=df['value'], mode='lines')

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="views")
fig.show()

Переименуем колонки для дальнейшей работы.

In [ ]:
df.columns = ['region', 'ds', 'metric', 'y']

df

,region,ds,metric,y
0,Kaliningrad,2019-01-01,views,26011.0
1,Kaliningrad,2019-01-02,views,23158.0
2,Kaliningrad,2019-01-03,views,23413.0
3,Kaliningrad,2019-01-04,views,22534.0
4,Kaliningrad,2019-01-05,views,21431.0
...,...,...,...,...
1456,Kaliningrad,2022-12-27,views,23796.0
1457,Kaliningrad,2022-12-28,views,22762.0
1458,Kaliningrad,2022-12-29,views,20338.0
1459,Kaliningrad,2022-12-30,views,10723.0


# 1 - Предобработка

Мы всегда должны начинать с предварительной обработки. Реальные данные могут содержать nan, аномалии и другие проблемы.

Для нашей задачи мы поработаем с nan.

## Задание 1.1 - Заполнение nan (0,5 балла)

🐴 **Найдите nan в датасете и заполните nan, используя подходящий метод.**

In [ ]:
# your code here
# ≽^•⩊•^≼

Нарисуем график временного ряда с заполненными nan.

In [ ]:
fig = px.line(title="BookBnb views for listings from Kaliningrad, daily")
fig.add_scatter(x=df.ds, y=df['y'], mode='lines')

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="views")
fig.show()

❓ **Почему вы выбрали этот способ заполнения nan? Напишите свои комментарии ниже:**

ваш ответ здесь



In [ ]:
# Проверка, что все ок

assert df.isna().sum().sum() == 0

# 2 - Регрессоры

Чтобы повысить точность вашего прогноза, вы будете использовать регрессоры, как фиктивные, так и непрерывные. Используя эти факторы, вы сможете лучше учитывать внешние события и тенденции, которые влияют на поведение пользователей на платформе.


Для решения этой задачи вы будете учитывать следующие регрессоры:

* **COVID-19**: период пандемии должен был повлиять на поведение путешественников и интерес к краткосрочной аренде жилья.

* **Введение новой фичи продукта**: в начале 2022 года на платформе был представлен новый алгоритм поиска, который, возможно, повлиял на вовлеченность пользователей.

* **DAU** (Ежедневные активные пользователи) платформы: общее количество активных пользователей на платформе каждый день, которое дает представление об общей активности и вовлеченности пользователей.

Эти регрессоры помогут вам охватить более широкий контекст и основные факторы, влияющие на метрику просмотров (views), что сделает ваш прогноз более надежным и содержательным.

В результате выполнения этой части экзамена вы создадите 2 датафрейма:

* **regressors** - датафрйем с вашими регрессорами за период вашего вр. ряда (с 2019 по конец 2022 года);
* **regressors_future** - датафрйем с вашими регрессорами для будущих дат (будущие *FORECAST_HORIZON* дней).

In [ ]:
# Создание датафрейма regressors

regressors = pd.DataFrame()
regressors['ds'] = df.ds
regressors

,ds
0,2019-01-01
1,2019-01-02
2,2019-01-03
3,2019-01-04
4,2019-01-05
...,...
1456,2022-12-27
1457,2022-12-28
1458,2022-12-29
1459,2022-12-30


In [ ]:
# Создание датафрейма regressors_future

regressors_future = pd.DataFrame()
regressors_future['ds'] = pd.date_range(start=df.ds.max()+ np.timedelta64(1, 'D'), periods=FORECAST_HORIZON)
regressors_future

,ds
0,2023-01-01
1,2023-01-02
2,2023-01-03
3,2023-01-04
4,2023-01-05
...,...
360,2023-12-27
361,2023-12-28
362,2023-12-29
363,2023-12-30


## 2.1 - Dummy регрессоры

Пришло время подготовить dummy регрессоры для COVID-19 и введения новой продуктовой фичи.

### Задание 2.1 - Подготовка dummy регрессоров (1 балл)

1) Давайте возьмем такие даты для COVID-19: с 30 января 2020 года по 5 мая 2023 года.

🐴 **Подготовьте dummy регрессор COVID-19 и добавьте его в датафреймы *regressors* и *regressors_future*.**

In [ ]:
# your code here
# ≽^•⩊•^≼

regressors['covid'] = # your code here
regressors_future['covid'] = # your code here

2) Новый алгоритм поиска был добавлен на платформу 1 января 2022.

🐴 **Подготовьте dummy регрессор для новой продуктовой фичи и добавьте его в датафреймы *regressors* и *regressors_future*.**

In [ ]:
# your code here
# ≽^•⩊•^≼

regressors['new_feature'] = # your code here
regressors_future['new_feature'] = # your code here

## 2.2 - Непрерывный регрессор DAU

*DAU* должен быть сильным регрессором для прогнозирования *views*, поскольку он отражает общий уровень активности на платформе. Увеличение *DAU* часто указывает на то, что больше пользователей активно используют приложение, что приводит к повышению активности поиска и увеличению количества просмотров объектов недвижимости. Эта связь делает DAU хорошим индикатором спроса и интереса пользователей, улавливающим поведенческие тенденции, такие как всплески во время праздников или после окончания маркетинговых кампаний.

Вы можете загрузить датасет *exam_dau_total.csv* с временными рядами для *DAU* по [ссылке](https://drive.google.com/file/d/11ScbYHtuXlG4siBDw5p9aDN47l6OrMH3/view?usp=sharing).

Набор данных содержит исторические данные **за период с начала 2019 года по конец 2022 года**. Обратите внимание, что это DAU для всей платформы.

**Колонки:**

* *dt* - даты, подневно, без пропусков;

* *metric* - название метрики, в этом датасете есть только "DAU";

* *value* - значение метрики.

In [ ]:
reg = pd.read_csv('exam_dau_total.csv', sep=';', parse_dates=['dt'])
reg

,dt,metric,value
0,2019-01-01,DAU,1541086.0
1,2019-01-02,DAU,1766712.0
2,2019-01-03,DAU,1800857.0
3,2019-01-04,DAU,1832445.0
4,2019-01-05,DAU,1871104.0
...,...,...,...
1456,2022-12-27,DAU,2357898.0
1457,2022-12-28,DAU,2305066.0
1458,2022-12-29,DAU,2293198.0
1459,2022-12-30,DAU,2243752.0


Давайте построим график для регрессора.

In [ ]:
fig = px.line(title="BookBnb DAU, daily")
fig.add_scatter(x=reg.dt, y=reg['value'], mode='lines')

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="DAU")
fig.show()

Не забывайте, что обычно вам также приходится предварительно обрабатывать и регрессоры.

🐴 **Проверьте регрессоры на наличие nan и используйте подходящий метод для их заполнения.**

In [ ]:
# your code here
# ≽^•⩊•^≼

Построим график для регрессора *DAU* с заполненными nan.

In [ ]:
fig = px.line(title="BookBnb DAU, daily")
fig.add_scatter(x=reg.dt, y=reg['value'], mode='lines')

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="DAU")
fig.show()

Основная проблема с этим регрессором заключается в том, что он отражает *DAU* для всей платформы *BookBnb* и является довольно шумным. Так, давайте не будем включать весь временной ряд *DAU*, а выделим тренд и используем его в качестве регрессора.

### Задание 2.2.1 - Определение тренда (0,5 балла)

Компоненты временного ряда могут быть извлечены с помощью соответствующего алгоритма декомпозиции.

🐴 **Извлеките тренд с помощью подходящего метода декомпозиции.**

In [ ]:
# your code here
# ≽^•⩊•^≼

model = # your code here
res = model.fit()

fig = res.plot()
fig.set_size_inches(10, 6)
plt.show()

❓ **Почему вы выбрали именно этот метод декомпозиции и такую/такие длины сезонного периода/периодов? Напишите свои комментарии ниже:**

ваш ответ здесь

🐴 **Проанализируйте остатки на автокорреляцию (с помощью 1 теста) и стационарность (с помощью 2 тестов).**

In [ ]:
# your code here
# ≽^•⩊•^≼

❓ **Успешна ли ваша декомпозиция судя по остаткам? Каковы могут быть причины такого результата? Напишите свои комментарии ниже:**

ваш ответ здесь

🐴 **Сохраните компоненту тренла из вашей декомпозиции в переменную trend_reg.**

In [ ]:
trend_reg = # your code here

С помощью приведенного ниже кода вы можете увидеть, как регрессор *тренда DAU* "коррелирует" с временными рядами просмотров.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.ds,
    y=df['y'],
    mode='lines',
    name='y',
    line=dict(color='blue'),
    yaxis='y1'
))

fig.add_trace(go.Scatter(
    x=trend_reg.dt, # или trend_reg.index
    y=trend_reg,
    mode='lines',
    name='net income',
    line=dict(color='red'),
    yaxis='y2'
))


fig.update_layout(
    title="BookBnb views for listings from Kaliningrad, daily",
    template='plotly_white',
    width=1000,
    height=500,
    yaxis=dict(
        title='stock prices',
    ),
    yaxis2=dict(
        title='net income',
        overlaying='y',
        side='right'
    ),
    xaxis=dict(title='Date')
)

fig.show()

### Задание 2.2.2 - Прогнозирование тренда с помощью Auto-ARIMAX (1 балл)

После выделения трендовой составляющей DAU следующим шагом является моделирование и прогнозирование этого тренда на будущие даты.

Поскольку тренды часто демонстрируют автокорреляцию и постепенные изменения во времени, статистическая модель, подобная Auto-ARIMA, хорошо подходит для этой задачи.

Поскольку COVID-19 повлиял на большинство бизнесов, вам также следует добавить dummy регрессор COVID и построить модель Auto-ARIMAX.

🐴 **Постройте модель Auto-ARIMAX (из `pmdarima`), которая учитывает регрессор COVID-19.**

In [ ]:
# your code here
# ≽^•⩊•^≼

model = # your code here

❓ **Как auto-ARIMA выбирает подходящие порядки (p, d, q)? Напишите свои комментарии ниже:**

ваш ответ здесь

Давайте посмотрим на описание получившейся модели.

In [ ]:
print(model.summary())

❓ **Напишите уравнение для построенной модели Auto-ARIMAX на основе описания (summary):**

ваш ответ здесь


🐴 **Постройте out-of-sample прогноз на будущие даты на горизонте FORECAST_HORIZON.**

In [ ]:
# your code here
# ≽^•⩊•^≼

trend_forecast = # your code here

Давайте построим график тренда и его Auto-ARIMA прогноза.

In [ ]:
fig = px.line(title="BookBnb DAU trend, daily")
fig.add_scatter(x=trend_reg.index, y=trend_reg, mode='lines', name='DAU', line=dict(color='blue'))
fig.add_scatter(x=trend_forecast.index, y=trend_forecast, mode='lines', name='DAU forecast', line=dict(color='red'))

fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="DAU")
fig.show()

🐴 **Добавьте *trend_reg* в датафрейм *regressors* и *trend_forecast* в датафрейм *regressors_future*.**

In [ ]:
# your code here
# ≽^•⩊•^≼

regressors['DAU'] = # your code here
regressors_future['DAU'] = # your code here

In [ ]:
# Проверка, что все ок

assert len(regressors.columns) == 4
assert len(regressors_future.columns) == 4

assert regressors.isna().sum().sum() == 0
assert regressors_future.isna().sum().sum() == 0

## Подготовка train-test

Теперь мы можем перейти к подготовке нашего изначального датасета для прогнозных моделей.

🐴 **Добавьте подготовленные регрессоры к исходному временному ряду.** Прогнозные модели , с которыми вы будете работать сегодня, будут забирать регрессоры из колонок подаваемых на вход данных.

In [ ]:
# your code here
# ≽^•⩊•^≼

df_with_reg = # your code here

Давайте разделим временной ряд *views* на train/test: 2019-2021 годы для train и 2022 год - для test.

🐴 **Разделите прогнозируемый временной ряд на train/test: 2019-2021 для train и 2022 - для test. Определите горизонт прогнозирования.**

In [ ]:
# your code here
# ≽^•⩊•^≼

train_size = # your code here
train, test = df_with_reg[:train_size], df_with_reg[train_size:]

In [ ]:
# your code here
# ≽^•⩊•^≼

forecast_horizon = # your code here
forecast_horizon

In [ ]:
# Проверка, что все ок

assert train.shape == (1096, 7)
assert test.shape == (365, 7)

Давайте построим разделение train/test.

In [ ]:
fig = px.line(title="train-test")
fig.add_scatter(x=train['ds'], y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test['ds'], y=test['y'], mode='lines', name='test', line=dict(color='green'))

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

# 3 - Прогнозирование моделью Prophet

Наконец, мы можем перейти к моделям. Для прогнозирования временного ряда *views* мы будем использовать модель Prophet. Prophet - это инструмент ML-прогнозирования, разработанный для эффективной работы с временными рядами, которые демонстрируют сезонность, тренд и влияние праздников.

Однако, прежде чем начать, вам следует выбрать подходящую метрику качества.

## Задание 3.1 - Выбор метрики качества (0,5 балла)

❓ **Какую метрику качества вы выбрали для выполнения задачи? Почему? Напишите свои комментарии ниже:**

ваш ответ здесь

## Task 3.2 - Модель Prophet

Вам также следует добавить в свою модель праздники и регрессоры. Prophet рассматривает оба этих компонента отдельно, и их добавление может улучшить ваш прогноз.

### Задание 3.2.1 - Праздники и регрессоры (0,5 балла)

🐴 **Подготовьте датафрейм *holidays_df* для учета праздников в вашей модели Prophet.**

In [ ]:
# Вот список российских праздников, которые вам следует учесть

russian_holidays = {
    "New Year": "01-01",
    "Christmas": "01-07",
    "Defender of the Fatherland Day": "02-23",
    "International Women's Day": "03-08",
    "Spring and Labour Day": "05-01",
    "Victory Day": "05-09",
    "Day of Russia": "06-12",
    "National Unity Day": "11-04"
}

In [ ]:
dates = []
holidays = []

for year in pd.date_range(start=df.ds.min(), end=df.ds.max(), freq='YS'):
  for holiday in russian_holidays.keys():
    dates.append(str(year.year)+'-'+russian_holidays[holiday])
    holidays.append(holiday)

In [ ]:
# your code here
# ≽^•⩊•^≼

holidays_df = # your code here

Обычно новогодний эффект присутствует за 10 дней до и после 1 января.

🐴 **Добавьте колонки для нижнего и верхнего окна к новогодним праздникам.** Подсказка: подбробнее изучите, как модель Prophet работает с праздниками.

In [ ]:
# your code here
# ≽^•⩊•^≼

In [ ]:
# Проверка, что все ок
assert holidays_df.shape == (32, 4)

### Задание 3.2.2 - Построение модели Prophet (0,5 балла)

🐴 **Создайте и обучите модель Prophet (из `prophet`) с аддитивной еженедельной сезонностью и мультипликативной годовой сезонности. Добавьте праздники и все регрессоры, рассмотренные в части 2. Постройте прогноз.**

In [ ]:
# your code here
# ≽^•⩊•^≼

model_prophet = # your code here

# your code here

In [ ]:
# Постройте прогноз
forecast_prophet = # your code here

Нарисуйте график прогноза.

In [ ]:
fig = px.line(title="BookBnb views for listings from Kaliningrad forecast, daily")
fig.add_scatter(x=train['ds'], y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test['ds'], y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=here['ds'], y=here['yhat'], mode='lines', name='forecast', line=dict(color='red'))


fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="views")
fig.show()

🐴 ** Оцените свой прогноз с помощью выбранной метрики качества.**

In [ ]:
# your code here
# ≽^•⩊•^≼

Далее, используйте модель с регрессорами и праздниками.

## Задание 3.3 - Тюнинг гиперпараметров (1 балл)

Как модель прогнозирования ML, Prophet требует точной настройки гиперпараметров. Далее вы будете настраивать гиперпараметры с помощью GridSearch + кросс-валидация.

🐴 ** Подготовьте сетку гиперпараметров.** Ваша сетка должна включать параметры для тренда, сезонности и праздников.

In [ ]:
# your code here
# ≽^•⩊•^≼

param_grid = {
    # your code for the grid here
}

# Создание всех комбинаций гипер-параметров
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]

len(all_params)

🐴 **Выполните тюнинг с помощью GridSearch + кросс-валидация.**

In [ ]:
# your code here
# ≽^•⩊•^≼

🐴 **Извлеките гиперпараметры, которые обеспечивают наилучшее качество модели.**

In [ ]:
# your code here
# ≽^•⩊•^≼

best_params = # your code here
print(best_params)

🐴 **Постройте, обучите модель и спрогнозируйте с помощью затюненной модели Prophet.**

In [ ]:
# your code here
# ≽^•⩊•^≼

model_prophet_tuned = # your code here

# your code here

In [ ]:
# Постройте прогноз
forecast_prophet_tuned = # your code here

Постройте график прогноза.

In [ ]:
fig = px.line(title="BookBnb views for listings from Kaliningrad forecast, daily")
fig.add_scatter(x=train['ds'], y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test['ds'], y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=forecast_prophet_tuned['ds'], y=forecast_prophet_tuned['yhat'], mode='lines', name='forecast', line=dict(color='violet'))


fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="views")
fig.show()

🐴 **Оцените ваш прогноз с помощью выбранно метрики качества.**

In [ ]:
# your code here
# ≽^•⩊•^≼

# 4 - Прогнозирование моделью MSTL

Теперь давайте перейдем к MSTL (Multiplicative Seasonal-Trend Decomposition using LOESS), статистической модели прогнозирования. В этой части вы будете использовать MSTL в качестве инструмента прогнозирования.

## Задание 4.1 - Модель MSTL (1 балл)

🐴 **Постройте, обучите модель и спрогнозируйте с помощью модели MSTL (из `statsforecast`) с регрессорами, рассмотренными в части 2.**

In [ ]:
# your code here
# ≽^•⩊•^≼

❓ **Почему вы выбрали такую/такие длину/длины для сезонного периода / периодов? Напишите свои комментарии ниже:**

ваш ответ здесь

In [ ]:
# Постройте прогноз
forecast_mstl = # your code here

In [ ]:
fig = px.line(title="BookBnb views for listings from Kaliningrad forecast, daily")
fig.add_scatter(x=train['ds'], y=(train['y']), mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test['ds'], y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=forecast_mstl['ds'], y=(forecast_mstl['MSTL']), mode='lines', name='forecast', line=dict(color='red'))

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="views")
fig.show()

🐴 **Оцените ваш прогноз с помощью выбранной метрики качества.**

In [ ]:
# your code here
# ≽^•⩊•^≼

# 5 - Сравнение моделей

Поскольку вы создали 2 разные модели, теперь пришло время выбрать лучшую из них. Для этого вам следует выполнить перекрестную проверку (CV), чтобы должным образом оценить вашу модель. Обратите внимание, что если вы используете разные библиотеки для перекрестной проверки, очень важно получить аналогичные фолды (cutoff) для ваших моделей.

## Задание 5.1 - Оцените свою лучшую модель Prophet с помощью CV (1 балл)

In [ ]:
# your code here
# ≽^•⩊•^≼

df_cv_prophet = # your code here to prepare a dataset with all the cutoffs of CV

In [ ]:
cv_metrics_prophet = # your code here

## Задание 5.2 - Оцените свою лучшую модель MSTL с помощью CV (1 балл)

In [ ]:
# your code here
# ≽^•⩊•^≼

df_cv_mstl = # your code here to prepare a dataset with all the cutoffs of CV

In [ ]:
cv_metrics_mstl = # your code here

In [ ]:
# Проверка, что все ок

assert list(df_cv_prophet.cutoff.unique()) == list(df_cv_mstl.cutoff.unique())

## Задание 5.3 - Лучшая модель (0,5 балла)

❓ **Почему вы выбрали именно такие параметры для кросс-валидаций? Напишите свои комментарии ниже:**

ваш ответ здесь

❓ **Какая модель лучшая? Как вы выбрали лучшую модель? Как вы думаете, почему именно эта модель оказалась лучшей? Напишите свои комментарии ниже:**

ваш ответ здесь

# 6 - Прогнозы

Наконец, после того, как модель была протестирована и показала себя превосходно, мы готовы представить прогноз на будущий год.

## Задание 6.1 - Out-of-sample прогноз с лучшей моделью (1 балл)

🐴 **Разработайте свою лучшую модель прогнозирования на основе полного временного ряда. Постройте out-of-sample прогноз на 1 будущий год.**

In [ ]:
# your code here
# ≽^•⩊•^≼

best_model = # your code here

# your code here

In [ ]:
# Постройте прогноз
forecast_future = # your code here

Постройте график прогноза.

In [ ]:
fig = px.line(title="BookBnb views for listings from Kaliningrad forecast, daily")
fig.add_scatter(x=train['ds'], y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test['ds'], y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=forecast_future['ds'], y=forecast_future['yhat'], mode='lines', name='forecast', line=dict(color='violet'))


fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="views")
fig.show()

**На этом все, поздравляю** ٩(⁎❛ᴗ❛⁎)۶